## Use CHOIR to find optimal resolution of clustering (in R)

- last updated: 04/21/2025
- https://www.choirclustering.com/articles/atlas_scale_data.html 

In [1]:
# load other libraries
# library(cicero)
library(Signac)
library(Seurat)
library(SeuratWrappers)
library(SeuratDisk) # to convert the h5ad to h5Seurat
library(dplyr)
library(Matrix)

.libPaths("/hpc/scratch/group.data.science/yangjoon.kim/.local/R_lib")
withr::with_libpaths(new = "/hpc/scratch/group.data.science/yangjoon.kim/.local/R_lib", library(CHOIR))
# withr::with_libpaths(new = "/hpc/scratch/group.data.science/yangjoon.kim/.local/R_lib", library(cicero))

Attaching SeuratObject

Registered S3 method overwritten by 'SeuratDisk':
  method            from  
  as.sparse.H5Group Seurat


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


                           ____
                         /  .-. \
                         \  \ / /
    _____  __    __       /\ \/        __   _____
  /      ||  |  |  |    /  /\ \       |  | |   _  \
 |  ,----'|  |__|  |  /  /  .----.    |  | |  |_|  |
 |  |     |   __   | |  |  / .---. \  |  | |      /
 |  `----.|  |  |  | |  |  `-' \ \' | |  | |  |\  \__
  \______||__|  |__|  `. ` .____| |/  |__| | _| `.___|
                         ` -----| |
                         /`.___/ /
                         `------'


CHOIR : Version 0.3.0
For more information see our website : www.CHOIRclustering.com
If you encounter a bug please report : https://github.com/Corc

In [2]:
# import the counts and metadata from the h5ad (exported)
filepath = "/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/annotated_data/objects_v2/peaks_by_pb_leiden_0.4_counts/"

obs_df <- read.csv(paste0(filepath, "/obs.csv"), row.names = 1)
var_df <- read.csv(paste0(filepath, "/var.csv"), row.names = 1)
mat <- readMM(paste0(filepath, "/sparse_matrix.mtx"))
mat <- t(mat)
# add the row and column names
colnames(mat) <- rownames(obs_df)
rownames(mat) <- rownames(var_df)

In [3]:
# Then create a Seurat object:
seurat <- CreateSeuratObject(counts = mat, meta.data = obs_df, assay="peaks_integrated")
seurat


Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


An object of class Seurat 
190 features across 640834 samples within 1 assay 
Active assay: peaks_integrated (190 features, 0 variable features)

In [5]:
# save the seurat object
saveRDS(seurat, file = paste0(filepath, "peaks_by_pseudobulks_leiden_0.4.rds"))

In [7]:
seurat@assays$peaks_integrated@data

An object of class Seurat 
190 features across 640834 samples within 1 assay 
Active assay: peaks_integrated (190 features, 0 variable features)

In [8]:
# run CHOIR (build and prune trees)
seurat <- CHOIR(seurat, n_cores=8, use_assay="peaks_integrated", use_slot="data")
seurat

----------------------------------------

- CHOIR - Part 1: Build clustering tree

----------------------------------------

2025-04-21 09:07:05 AM : (Step 1/7) Checking inputs and preparing object..


Input data:
 - Object type: Seurat (v4)
 - # of cells: 640834
 - # of batches: 1
 - # of modalities: 1
 - ATAC data: FALSE
 - Countsplitting: FALSE
 - Assay: peaks_integrated
 - Slot used to build tree: data
 - Slot used to prune tree: data


Proceeding with the following parameters:
 - Intermediate data stored under key: CHOIR
 - Alpha: 0.05
 - Multiple comparison adjustment: bonferroni
 - Features to train RF: var
 - # of excluded features: 0
 - # of permutations: 100
 - # of RF trees: 50
 - Use variance: TRUE
 - Minimum accuracy: 0.5
 - Minimum connections: 1
 - Maximum repeated errors: 20
 - Distance approximation: TRUE
 - Maximum cells sampled: Inf
 - Downsampling rate: 0.116
 - Minimum reads: >0 reads
 - Maximum clusters: auto
 - Minimum cluster depth: 2000
 - Normalization method:

## testing the atlas-scale CHOIR run
- use subset to parallelize the hierarchical clustering


In [ ]:
# Step 1. Generate parent clusters
seurat <- buildParentTree(seurat, use_assay="peaks_integrated", 
                         use_slot="data", n_cores=8,
                         cluster_params = list(algorithm = 4, group.singletons = TRUE))
seurat


Input data:
 - Object type: Seurat (v4)
 - # of cells: 640834
 - # of batches: 1
 - # of modalities: 1
 - ATAC data: FALSE
 - Countsplitting: FALSE
 - Assay: peaks_integrated
 - Slot used to build tree: data
 - Slot used to prune tree: data


Proceeding with the following parameters:
 - Intermediate data stored under key: CHOIR
 - Distance approximation: TRUE
 - Downsampling rate: 0.116
 - Normalization method: none
 - Dimensionality reduction method: Default
 - Dimensionality reduction parameters provided: No
 - # of variable features: Default
 - Batch correction method: none
 - Batch correction parameters provided: No
 - Nearest neighbor parameters provided: 
     - verbose: FALSE
 - Clustering parameters provided: 
     - algorithm: 4
     - group.singletons: TRUE
     - verbose: FALSE
 - # of cores: 8
 - Random seed: 1


2025-04-21 10:27:02 AM : (Step 1/4) Running initial dimensionality reduction..

2025-04-21 10:27:02 AM : Preparing matrix using 'peaks_integrated' assay and 'data

In [ ]:
# Step 2. Subset each parent cluster
# Suppose we look at unique parent clusters
parent_clusters <- unique(seurat$CHOIR_parent_clusters)

# For each parent cluster, subset the Seurat object
subtree_objects_list <- list()
for (pc in parent_clusters) {
  subtree_s <- subset(seurat, subset = CHOIR_parent_clusters == pc)
#   # If the subset is still too large (> 450k cells), do minimal splits
#   # (Your data has only 150 columns, so likely no need, but here’s how):
#   if (ncol(subtree_s) > 450000) {
#     # Identify subclusters at a starting resolution, then subset further
#     subtree_s <- subtree_s %>% 
#       FindVariableFeatures() %>% 
#       ScaleData() %>% 
#       RunPCA() %>%
#       FindNeighbors()
    
#     # The internal CHOIR function to guess a good starting resolution
#     starting_resolution <- CHOIR:::.getStartingResolution(
#       subtree_s@graphs$RNA_snn,
#       cluster_params = list(algorithm = 1, group.singletons = TRUE),
#       random_seed = 1, 
#       verbose = TRUE
#     )
    
#     subtree_s <- FindClusters(
#       subtree_s, 
#       resolution = starting_resolution[["starting_resolution"]]
#     )
    
#     # You might now subset again for each new cluster (0, 1, 2, etc.)
#     # e.g., subtree_s_0 <- subset(subtree_s, subset = seurat_clusters == 0)
#     # subtree_s_1 <- subset(subtree_s, subset = seurat_clusters == 1)
#     # ...
#     # Then store them in subtree_objects_list
#   }
  
  subtree_objects_list[[pc]] <- subtree_s
}

subtree_records_list <- list()

for (pc in names(subtree_objects_list)) {
  subtree_s <- subtree_objects_list[[pc]]
  
  if (ncol(subtree_s) > 1) {
    # Call CHOIR with the same downsampling_rate you used in buildParentTree
    subtree_s <- CHOIR(
      subtree_s,
      key = "CHOIR_subtree",
      downsampling_rate = seurat@misc$CHOIR$parameters$buildParentTree_parameters$downsampling_rate
      # Additional params as needed
    )
  } else {
    # If there's effectively 1 or 0 columns, set cluster label manually
    subtree_s@misc$CHOIR_subtree$clusters$CHOIR_clusters_0.05 <- data.frame(
      CellID = colnames(subtree_s),
      CHOIR_clusters_0.05 = 1,
      Record_cluster_label = "P0_L0_1"
    )
    subtree_s@misc$CHOIR_subtree$parameters <- seurat@misc$CHOIR$parameters
  }
  
  # Save back
  subtree_objects_list[[pc]] <- subtree_s
  subtree_records_list[[pc]] <- getRecords(subtree_s, key = "CHOIR_subtree")
}

In [ ]:
# Step 4. Combine subtrees and standardize significance thresholds
# Run the CHOIR function combineTrees on the complete set of records extracted in step 2. In doing so, the significance threshold will be standardized across all clustering trees, yielding a final set of clusters.
seurat <- combineTrees(seurat,
                       subtree_list = subtree_records_list)